# Parallel Batch Motion Correction Demo (TIFF + Binning)

This notebook demonstrates how to process `.tif` microscopy files using a parallelized bin-register-interpolate-apply workflow based on Pystackreg.

Requires installation of: pystackreg, zarr and joblib

- Loads `.tif` files (TIFF stacks)
- Temporally bins the data for robust registration
- Calculates motion correction transforms on binned data
- Interpolates transforms to full frame rate
- Applies transforms to original data
- Saves output as `.tif` 

In [5]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import sys

sys.path.append('../../src')

from parallel_motion_correct import (
    parallel_batch_process,
    find_files,
    get_available_cores,
    load_data
)

print(f"Available CPU cores: {get_available_cores()}")

Available CPU cores: 24


## 1. Setup Data Paths
Configure where to find the raw files.

In [6]:
# Option 1: Load from CSV (User legacy workflow)
file_list = []

try:
    csv_path = '../../../Ryr1-Myo15GCamp.xlsx'
    if os.path.exists(csv_path):
        allData = pd.read_excel(csv_path)
        # Note: Adjust 'drive' and path mapping if running on Linux with Windows paths in CSV
        drive = 'Z' 
        allData['Folder'] = drive + allData['Folder'].str[1:]

        fileName = '1-jumpCorrected.tif'
        
        # Collect potential paths
        for f in allData['Folder'].values:
            if isinstance(f, str):
                # If paths in CSV are absolute Windows paths, you might need to fix them manually here
                # Here we assume they might be relative or correctable
                path = Path(f) / 'processedMovies' / fileName
                if path.exists():
                    file_list.append(path)
                    
        print(f"Found {len(file_list)} existing files from CSV")
    else:
        print(f"CSV {csv_path} not found, skipping csv loading.")
except Exception as e:
    print(f"Error loading from CSV: {e}")

# Option 2: Search directory (Fallback) - Now searches for .tif files
if not file_list:
    search_dir = Path('.') # Current directory or replace with data path
    print(f"Searching {search_dir.resolve()} for .tif files...")
    # Search for TIFF files instead of RAW files
    file_list = list(search_dir.rglob('*.tif')) + list(search_dir.rglob('*.tiff'))
    print(f"Found {len(file_list)} TIFF files in directory search")
    
# Display found files
for f in file_list[:5]:
    print(f"  - {f}")
if len(file_list) > 5:
    print(f"  ... and {len(file_list)-5} more")

CSV ../../../Ryr1-Myo15GCamp.xlsx not found, skipping csv loading.
Searching /home/marcotti-lab/git-repos/In-Vivo-Analysis-core/notebooks/1-MotionCorrection and segmentation for .tif files...
Found 0 TIFF files in directory search


## 2. Configure Processing Parameters

In [7]:
config = {
    'output_dir':None, # Base output dir (Note: actual files saved in 'corrected' subfolder of input)
    'transform_type': 'rigid',     # 'translation', 'rigid', 'scaled', 'affine', 'bilinear'
    'reference_frames': 50,        # Frames to average for reference image
    'temporal_bin': 10,            # Frames to average together for robust registration
    'n_frames': None,              # Process all frames (None)
    'save_transforms': False,       # Save transformation matrices as .npy
    'n_jobs': 15,                  # Use all available cores
    'apply_to_full': True,          # False: save the binned, True: save the original file using the interpolated transformation from the binned file. 
    'save_binned' : False,           # Save also the binned file if apply_to_full is True
    'chunk_size' : 3000,
    'skip_existing': True,      # ← Add this
    'suffix' : '-mc',
    'correct_intensity': True,    # Apply intensity correction to avoid brightness shifs

}

print("Configuration:")
for k, v in config.items():
    print(f"  {k}: {v}")

Configuration:
  output_dir: None
  transform_type: rigid
  reference_frames: 50
  temporal_bin: 10
  n_frames: None
  save_transforms: False
  n_jobs: 15
  apply_to_full: True
  save_binned: False
  chunk_size: 3000
  skip_existing: True
  suffix: -mc
  correct_intensity: True


## 3. Run Parallel Processing

In [8]:
if file_list:
    results = parallel_batch_process(
        file_list=file_list,
        **config
    )
else:
    print("No files to process!")
    results = []

No files to process!


## 4. Summary

In [9]:
print(f"{'='*80}")
print("PROCESSING SUMMARY")
print(f"{'='*80}")
print(f"{'File':<40} {'Status':<10} {'Frames':<10} {'Time (s)':<10}")
print(f"{'-'*80}")

successful = []
failed = []

for r in results:
    status = "✓" if r.success else "✗"
    name = r.input_path.name[:38]
    print(f"{name:<40} {status:<10} {r.n_frames:<10} {r.processing_time:<10.1f}")
    if r.success:
        successful.append(r)
    else:
        failed.append(r)

print(f"{'-'*80}")
print(f"Total: {len(successful)} succeeded, {len(failed)} failed")

if failed:
    print("\nErrors:")
    for r in failed:
        print(f"{r.input_path}: {r.error_message}")

PROCESSING SUMMARY
File                                     Status     Frames     Time (s)  
--------------------------------------------------------------------------------
--------------------------------------------------------------------------------
Total: 0 succeeded, 0 failed
